In [1]:
from cgra import *
from kernels import *
from sat_to_csv import *

In [2]:
kernel_name = "add_vectors_v1"
version = "_vector_addition_autogenerated"

In [3]:
# Global variables
CGRB_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [4]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [5]:
# Data
def configMemory(A_data, B_data, vlen):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # nIt           &A[0]         &A[1]         &A[2]
    # &A[3]         &A[4]         &A[5]         &A[6]
    # &A[7]         &A[8]         &A[9]         &A[10]
    # &A[11]        &A[12]        &A[13]        &A[14]
    # ----------------------
    # -             &B[0]         &B[1]         &B[2]
    # &B[3]         &B[4]         &B[5]         &B[6]
    # &B[7]         &B[8]         &B[9]         &B[10]
    # &B[11]        &B[12]        &B[13]        &B[14]
    # ----------------------
    # -             &C[0]         &C[1]         &C[2]
    # &C[3]         &C[4]         &C[5]         &C[6]
    # &C[7]         &C[8]         &C[9]         &C[10]
    # &C[11]        &C[12]        &C[13]        &C[14]
    
    first_addr_A = first_addr
    first_addr_B = first_addr_A + vlen*4
    first_addr_C = first_addr_B + vlen*4
    nIterations = int(vlen/15)

    config_vals_col0 = [nIterations, first_addr_A+3*4, first_addr_A+7*4, first_addr_A+11*4,
                        first_addr_B+3*4, first_addr_B+7*4, first_addr_B+11*4,
                        first_addr_C+3*4, first_addr_C+7*4, first_addr_C+11*4]
    config_vals_col1 = [first_addr_A, first_addr_A+4*4, first_addr_A+8*4, first_addr_A+12*4,
                        first_addr_B, first_addr_B+4*4, first_addr_B+8*4, first_addr_B+12*4,
                        first_addr_C, first_addr_C+4*4, first_addr_C+8*4, first_addr_C+12*4]
    config_vals_col2 = [first_addr_A+4, first_addr_A+5*4, first_addr_A+9*4, first_addr_A+13*4,
                        first_addr_B+4, first_addr_B+5*4, first_addr_B+9*4, first_addr_B+13*4,
                        first_addr_C+4, first_addr_C+5*4, first_addr_C+9*4, first_addr_C+13*4]
    config_vals_col3 = [first_addr_A+4*2, first_addr_A+6*4, first_addr_A+10*4, first_addr_A+14*4,
                        first_addr_B+4*2, first_addr_B+6*4, first_addr_B+10*4, first_addr_B+14*4,
                        first_addr_C+4*2, first_addr_C+6*4, first_addr_C+10*4, first_addr_C+14*4]
    
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [6]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [7]:
def getResult(first_addr_C, end_addr_C, vlen):
    result = [0 for _ in range(vlen)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [8]:
def add_vectors_cpu(A_data, B_data, vlen):
    expected_res = [0 for _ in range(vlen)]
    for i in range(vlen):
        expected_res[i] = A_data[i] + B_data[i]
    return expected_res

In [9]:
# Test dimensions (4xXx4)
VECTOR_SIZE = 5*15 # Multiplo de 15
A_data = list(range(0, VECTOR_SIZE))
B_data = [x + 100 for x in range(0, VECTOR_SIZE)]

load_addrs = configMemory(A_data, B_data, VECTOR_SIZE)

In [10]:
runKernel(load_addrs, max_it=200000)

Instr =  0 ( 0 )
[   5, 20000, 20004, 20008]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[20012, 20016, 20020, 20024]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[20028, 20032, 20036, 20040]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[20044, 20048, 20052, 20056]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
-------
Instr =  1 ( 1 )
[   0, 20300, 20304, 20308]    [SADD R1  0  0, LWD R1  4, LWD R1  4, LWD R1  4]    
[20312, 20316, 20320, 20324]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[20328, 20332, 20336, 20340]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[20344, 20348, 20352, 20356]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
-------
Instr =  2 ( 2 )
[   0, 20600, 20604, 20608]    [NOP , LWD R2  4, LWD R2  4, LWD R2  4]    
[20612, 20616, 20620, 20624]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[20628, 20632, 20636, 20640]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[20644, 20648, 20652, 20656]    [LWD R2  4, LW

In [11]:
# Get result from CGRA
first_addr_C = first_addr + VECTOR_SIZE*4*2
end_addr_C = first_addr_C + VECTOR_SIZE*4
result = getResult(first_addr_C, end_addr_C, VECTOR_SIZE)

# Get cpu output
expected_res = add_vectors_cpu(A_data, B_data, VECTOR_SIZE)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print(result)
    print(expected_res)
else:
    print("OK")



Err: 56
